# 🎙️ F5-TTS Batch Voice Cloning - Google Colab T4

Notebook ini khusus untuk menjalankan **F5-TTS Voice Cloning** secara otomatis (batch) untuk seluruh speaker di Google Drive.

### 📌 Fitur Utama:
1. **Recursive Reference Scan**: Otomatis mendeteksi seluruh file audio di folder `Dataskripsi` di Google Drive.
2. **Smart Resume**: Menyeberangi (skip) audio MP3 yang sudah berhasil di-generate di Drive.
3. **Automatic GPU Cleanup**: Bebas CUDA Memory Leak pada T4 GPU.
4. **Direct Drive Sync**: Hasil MP3 disalin langsung ke `/content/drive/MyDrive/data skirpsi/VoiceCloning/F5-TTS`.

In [ ]:
# Step 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Step 2: Install F5-TTS & Dependensi Audio
!pip install --no-build-isolation f5-tts ffmpeg-python soundfile pydub mecab-python3 unidic num2words pythainlp subword_nmt cached_path cn2an pypinyin pykakasi g2p_en g2pM fugashi ipadic unidic_lite anyascii jamo gruut gruut_ipa huggingface_hub


In [ ]:
# Step 3: Konfigurasi Teks dan Batch
SKIP_EXISTING = True  # Set True untuk melewatin audio yang sudah ada di Drive
BATCH_START = 0       # Indeks awal speaker
BATCH_COUNT = 100     # Jumlah maksimal speaker yang diproses

TEXTS = {
    'id': 'Indonesia adalah negara kepulauan yang kaya budaya, bahasa, sumber daya alam, serta memiliki masyarakat beragam yang hidup saling berdampingan harmonis.',
    'en': 'England is a historic European country known for its royal family, famous universities, football, diverse culture, and remarkable technological development worldwide.'
}
print('⚙️ Konfigurasi F5-TTS Siap.')


In [ ]:
# Step 4: Run Batch Generation F5-TTS
import os, sys, gc, shutil, subprocess
from pathlib import Path
import torch

ROOT = Path('/content')
REF_ROOT = ROOT / 'reference_dataskripsi'
OUT_DIR = ROOT / 'generated_voice_cloning' / 'F5-TTS'
DRIVE_OUT = Path('/content/drive/MyDrive/data skirpsi/VoiceCloning/F5-TTS')

REF_ROOT.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_OUT.mkdir(parents=True, exist_ok=True)

def run_cmd(*args):
    cmd = [str(a) for a in args]
    p = subprocess.run(cmd, capture_output=True, text=True)
    return p

def free_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

def scan_references():
    drive_folder = Path('/content/drive/MyDrive/Dataskripsi')
    if not drive_folder.exists():
        cands = list(Path('/content/drive/MyDrive').rglob('*Dataskripsi*'))
        if cands: drive_folder = cands[0]
        else: raise FileNotFoundError('Folder Dataskripsi tidak ditemukan di Google Drive!')
    
    print(f'📂 Scanning folder reference dari: {drive_folder}')
    audio_exts = {'.wav', '.mp3', '.m4a', '.ogg', '.flac'}
    copied = 0
    for f in drive_folder.rglob('*'):
        if f.is_file() and f.suffix.lower() in audio_exts:
            target_name = f.stem + '.wav' if f.suffix.lower() != '.wav' else f.name
            target_path = REF_ROOT / target_name
            if not target_path.exists():
                if f.suffix.lower() != '.wav':
                    run_cmd('ffmpeg', '-y', '-i', str(f), '-ar', '24000', '-ac', '1', str(target_path))
                else:
                    shutil.copy2(f, target_path)
                copied += 1
    print(f'  ✅ Prepared {copied} new audio file(s) di {REF_ROOT}')
    
    speaker_map = {}
    for p in REF_ROOT.glob('*.wav'):
        name = p.stem
        spk_id = name
        lang = 'id'
        if name.startswith('Suaraindo_'):
            spk_id = name[len('Suaraindo_'):]
            lang = 'id'
        elif name.startswith('Suaraeng_'):
            spk_id = name[len('Suaraeng_'):]
            lang = 'en'
        elif '_' in name:
            spk_id = name.split('_')[-1]
        
        if spk_id not in speaker_map: speaker_map[spk_id] = {'id': p, 'en': p}
        if lang == 'id': speaker_map[spk_id]['id'] = p
        elif lang == 'en': speaker_map[spk_id]['en'] = p
    return speaker_map

spk_map = scan_references()
all_speakers = sorted(list(spk_map.keys()))
selected_speakers = all_speakers[BATCH_START : BATCH_START + BATCH_COUNT]
print(f'🎙️ Total Speaker Terdeteksi: {len(all_speakers)} | Memproses Batch ({len(selected_speakers)} speaker): {selected_speakers}')

for spk in selected_speakers:
    ref_id = spk_map[spk]['id']
    ref_en = spk_map[spk]['en']
    for lang, text, ref in (('id', TEXTS['id'], ref_id), ('en', TEXTS['en'], ref_en)):
        final_mp3_name = f'{spk}_{lang}.mp3'
        drive_mp3 = DRIVE_OUT / final_mp3_name
        if SKIP_EXISTING and drive_mp3.exists() and drive_mp3.stat().st_size > 1000:
            print(f'  ⏩ [SKIP] {spk}_{lang} (Sudah ada di Drive)')
            continue
        
        print(f'  ⚡ [GENERATE] F5-TTS {spk}_{lang}...')
        wav_out = OUT_DIR / f'{spk}_{lang}.wav'
        mp3_out = OUT_DIR / final_mp3_name
        try:
            res = run_cmd('f5-tts_infer-cli', '--model', 'F5TTS_v1_Base', '--ref_audio', str(ref), '--ref_text', '', '--gen_text', text, '--output_dir', str(OUT_DIR), '--output_file', f'{spk}_{lang}.wav')
            if wav_out.exists():
                run_cmd('ffmpeg', '-y', '-i', str(wav_out), '-b:a', '192k', str(mp3_out))
                shutil.copy2(mp3_out, drive_mp3)
                print(f'  ☁️ Saved to Drive: {final_mp3_name}')
            else:
                print(f'  ❌ [ERROR] Output WAV tidak terbentuk: {res.stderr[:200]}')
        except Exception as e:
            print(f'  ❌ [ERROR] {spk}_{lang}: {e}')
        finally:
            free_memory()

print('🎉 F5-TTS BATCH GENERATION SELESAI!')
